# Phase 3 figures: hard viscosity ($\nu = 0.01/\pi$)

**WISER x BQP 2026, Track A | Burgers at the Shah/Raissi viscosity**

Reads the tables produced by `Phase3_Hard_InputBuilder.ipynb` and produces the complete figure
set for Workstreams A, B and C. **Nothing is recomputed and no synthetic data is used**: every
value plotted comes from those CSVs, which came from the trained models.

## A correction the builder's manifest forced

While writing the builder I assumed the QAPINN had trained $2.2\times$ longer than the classical
baselines at $q=3$--$q=7$. The recorded histories say otherwise: **every model at $q=3$--$q=7$
has 2016 recorded steps**, QAPINN, twin and GAAF alike. They ran the same curriculum
(2000 Adam iterations plus 2400 L-BFGS iterations recorded in chunks of 150). Only **$q=8$
differs, at 703 steps**, because of the taper applied to fit it in the compute budget.

The consequence is favourable and important: at $q=3$--$q=7$ the comparison between the quantum
model and its parameter-matched classical control is **budget-matched and therefore fair**. The
head-to-head spectral comparison, which had to be hedged at $\nu=0.05$, is clean here. Only $q=8$
is set aside, and it is marked `in_primary=False` in every table rather than deleted.

## Corpus

58 runs rebuilt from 116 checkpoints found (the remainder duplicates across merged folders),
zero rebuild failures, no dead runs. QAPINN, twin and GAAF each cover $n=3\dots8$ with three
seeds, plus a few GAAF retries. Spectra, activations and CKA are stored at 26 time slices.

## Figure families

**Workstream A** (does the quantum layer reach frequencies the classical one cannot?): the
high-frequency fraction against time with the analytical target overlaid, the recovery fraction
against width, the spectral centroid, and the seed spread.

**Workstream B** (what is lost): gradient variance against width, cost per step against width,
optimisation traces, accuracy, and a combined cost-benefit panel.

**Workstream C** (how information flows): CKA from the first layer to the output against time and
against width, CKA by depth, the layer-to-layer matrix that was deferred at $\nu=0.05$,
per-neuron spread, the neuron-against-time map, and saturation by layer.

Output goes to **`WISER Results/Phase 3 Hard/figures/`**.

## 1. Mount and load the tables

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, json, math
import numpy as np, pandas as pd

MYDRIVE = "/content/drive/MyDrive"
OUT = None
for c in sorted(glob.glob(os.path.join(MYDRIVE, "**", "Phase 3 Hard"), recursive=True)):
    if os.path.isdir(c) and os.path.exists(os.path.join(c, "wsa_hard_metrics_by_slice.csv")):
        OUT = c; break
assert OUT, "Phase 3 Hard folder with wsa_hard_metrics_by_slice.csv not found"
FIG = os.path.join(OUT, "figures"); os.makedirs(FIG, exist_ok=True)
print("input folder :", OUT)
print("figures      :", FIG)

WSA  = pd.read_csv(os.path.join(OUT, "wsa_hard_metrics_by_slice.csv"))
GRAD = pd.read_csv(os.path.join(OUT, "wsb_hard_gradient_variance.csv"))
TIME = pd.read_csv(os.path.join(OUT, "wsb_hard_timing.csv"))
HIST = pd.read_csv(os.path.join(OUT, "wsb_hard_loss_traces.csv"))
META = pd.read_csv(os.path.join(OUT, "run_metadata.csv"))
CKA  = pd.read_csv(os.path.join(OUT, "wsc_hard_cka_by_slice.csv"))
NEU  = pd.read_csv(os.path.join(OUT, "wsc_hard_neuron_stats.csv"))
MAN  = json.load(open(os.path.join(OUT, "phase3_hard_manifest.json")))

for nm, d in [("wsa", WSA), ("grad", GRAD), ("timing", TIME), ("hist", HIST),
              ("meta", META), ("cka", CKA), ("neuron", NEU)]:
    print("%-8s %8d rows" % (nm, len(d)))

ANCHOR = float(MAN["anchor_slice"])
PHI_MAX = float(MAN["analytical_phi_max"])
print("\nanchor slice t = %.2f, analytical phi_>8 there = %.4e" % (ANCHOR, PHI_MAX))
print("widths :", sorted(WSA.n_feat.unique()))
print("models :", sorted(WSA.model.unique()))
print("slices : %d" % WSA.t.nunique())

print("\nrecorded steps by model and width (the budget check):")
b = META.groupby(["model", "n_feat"]).budget_steps.agg(["min", "max", "count"])
print(b.to_string())
print("\nq8 is the only width whose budget differs; it is excluded from primary analysis.")

## 2. Style

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9.5, "axes.titlesize": 10.5,
    "axes.labelsize": 9.5, "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "legend.fontsize": 8.2, "legend.frameon": True, "legend.framealpha": 0.92,
    "legend.edgecolor": "0.8", "axes.grid": True, "grid.alpha": 0.20,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 120, "savefig.dpi": 300})

C_Q, C_T, C_G, C_REF = "#14285a", "#c0392b", "#2e8b57", "#000000"
STYLE = {"qapinn": (C_Q, "QAPINN", "o"),
         "twin":   (C_T, "c-PINN (classical twin)", "s"),
         "gaaf":   (C_G, "GAAF-PINN", "^")}
ORDER = ["qapinn", "twin", "gaaf"]
SUB = r"Burgers, $\nu = 0.01/\pi$"

def savefig(fig, name):
    p = os.path.join(FIG, name + ".png")
    fig.savefig(p, bbox_inches="tight"); plt.close(fig); print("[fig]", name + ".png")

P = WSA[WSA.in_primary == True].copy()          # q8 excluded
ALL = WSA.copy()
WID_P = sorted(P.n_feat.unique())
WID_A = sorted(ALL.n_feat.unique())
print("primary widths:", WID_P, " all widths:", WID_A)

# Three series at the same x overlap and become unreadable. Each series is drawn
# with a small horizontal offset, a distinct marker, and a white marker edge, so
# the points stay separable without changing any value.
DODGE = {"qapinn": -0.10, "twin": 0.0, "gaaf": +0.10}
MSTYLE = dict(ms=8, mew=1.2, mec="white", lw=1.9, capsize=3, capthick=1.1, elinewidth=1.1)

def dx(model):
    return DODGE.get(model, 0.0)

def series_plot(ax, x, y, yerr, model, label, log=False):
    col, _, mk = STYLE[model]
    xs = np.asarray(x, float) + dx(model)
    if yerr is None:
        ax.plot(xs, y, color=col, marker=mk, label=label, **{k: v for k, v in MSTYLE.items()
                                                             if k in ("ms", "mew", "mec", "lw")})
    else:
        ax.errorbar(xs, y, yerr=yerr, color=col, marker=mk, label=label, **MSTYLE)
    if log: ax.set_yscale("log")

def first_col(df):
    cols = [c for c in df.columns if c.startswith("cka_") and c.endswith("_out")
            and ("L0" in c or "quantum" in c or "front" in c)]
    s = pd.Series(np.nan, index=df.index)
    for c in cols: s = s.fillna(df[c])
    return s

## 3. Workstream A: does the quantum layer reach higher frequencies?

The measurement that was uninformative at $\nu=0.05$, where the target carried
$\varphi_{>8}\approx 10^{-6}$. Here the analytical target carries $1.88\times10^{-2}$, so the
metric has genuine signal to resolve.

In [ ]:
# ---- HA1: high-frequency fraction against time, per width, target overlaid -----
ncol = 3; nrow = math.ceil(len(WID_P)/ncol)
fig, axs = plt.subplots(nrow, ncol, figsize=(4.1*ncol, 3.2*nrow),
                        sharex=True, sharey=True, squeeze=False)
for i, n in enumerate(WID_P):
    ax = axs[i//ncol][i % ncol]
    ex = P[P.n_feat == n].groupby("t").phi_exact.first()
    ax.plot(ex.index, ex.values, color=C_REF, ls="--", lw=2, label="analytical target", zorder=5)
    for m in ORDER:
        s = P[(P.n_feat == n) & (P.model == m)]
        if s.empty: continue
        col, lab, _ = STYLE[m]
        for _, g in s.groupby("seed"):
            g = g.sort_values("t"); ax.plot(g.t, g.phi_model, color=col, alpha=0.6, lw=1.0)
        ax.plot([], [], color=col, label="%s (%d seeds)" % (lab, s.seed.nunique()))
    ax.set_title("$n=%d$" % n); ax.set_yscale("log")
axs[0][0].legend(loc="lower right", fontsize=7.2)
for j in range(ncol): axs[nrow-1][j].set_xlabel("$t$")
for i2 in range(nrow): axs[i2][0].set_ylabel(r"$\varphi_{>8}$")
for i in range(len(WID_P), nrow*ncol): axs[i//ncol][i % ncol].axis("off")
fig.suptitle("High-frequency content against time, with the analytical target\n%s" % SUB,
             y=1.005)
plt.tight_layout(); savefig(fig, "HA1_phi_vs_time_by_width")

# ---- HA2: recovery fraction against width -------------------------------------
A = P[np.isclose(P.t, ANCHOR)]
fig, axs = plt.subplots(1, 2, figsize=(11.2, 4.3))
ax = axs[0]
for m in ORDER:
    s = A[A.model == m]
    if s.empty: continue
    col, lab, mk = STYLE[m]
    g = s.groupby("n_feat").phi_recovery
    series_plot(ax, g.mean().index, g.mean().values, g.std().values, m, lab)
ax.axhline(1.0, color=C_REF, ls=":", lw=1.3, label="perfect recovery")
ax.set_xlabel("qubit count / feature width $n$")
ax.set_ylabel(r"recovery fraction $\varphi_{\mathrm{model}}/\varphi_{\mathrm{exact}}$")
ax.set_title("(a)  Recovery of true high-frequency content")
ax.set_xticks(WID_P); ax.legend(loc="best")

ax = axs[1]
for m in ORDER:
    s = A[A.model == m]
    if s.empty: continue
    col, lab, mk = STYLE[m]
    g = s.groupby("n_feat").phi_model
    series_plot(ax, g.mean().index, g.mean().values, g.std().values, m, lab)
ax.axhline(A.phi_exact.mean(), color=C_REF, ls="--", lw=1.4, label="analytical target")
ax.set_yscale("log"); ax.set_xlabel("qubit count / feature width $n$")
ax.set_ylabel(r"$\varphi_{>8}$")
ax.set_title(r"(b)  Absolute high-frequency content at $t=%.2f$" % ANCHOR)
ax.set_xticks(WID_P); ax.legend(loc="best")
fig.suptitle(SUB, y=1.02)
plt.tight_layout(); savefig(fig, "HA2_recovery_and_absolute")

# ---- HA3: spectral centroid ----------------------------------------------------
fig, axs = plt.subplots(1, 2, figsize=(11.2, 4.3))
ax = axs[0]
for m in ORDER:
    s = A[A.model == m]
    if s.empty: continue
    col, lab, mk = STYLE[m]
    g = s.groupby("n_feat").centroid_model
    series_plot(ax, g.mean().index, g.mean().values, g.std().values, m, lab)
ax.axhline(A.centroid_exact.mean(), color=C_REF, ls="--", lw=1.4, label="analytical target")
ax.set_xlabel("width $n$"); ax.set_ylabel(r"spectral centroid (cyc/$x$)")
ax.set_title("(a)  Centroid against width"); ax.set_xticks(WID_P)
ax.legend(loc="best"); ax.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))
ax = axs[1]
ex = P.groupby("t").centroid_exact.first()
ax.plot(ex.index, ex.values, color=C_REF, ls="--", lw=2, label="analytical target")
for m in ORDER:
    s = P[P.model == m]
    if s.empty: continue
    col, lab, mk = STYLE[m]
    g = s.groupby("t").centroid_model.mean()
    ax.plot(g.index, g.values, color=col, lw=1.7, label=lab)
ax.set_xlabel("$t$"); ax.set_ylabel(r"spectral centroid (cyc/$x$)")
ax.set_title("(b)  Centroid against time, mean over widths and seeds")
ax.legend(loc="best")
fig.suptitle(SUB, y=1.02)
plt.tight_layout(); savefig(fig, "HA3_centroid")

# ---- HA4: seed spread ----------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.2, 4.3))
for m in ORDER:
    s = A[A.model == m]
    if s.empty: continue
    col, lab, mk = STYLE[m]
    g = s.groupby("n_feat").phi_recovery.std()
    ax.plot(g.index, g.values, color=col, marker=mk, ms=6, lw=1.7, label=lab)
ax.set_xlabel("width $n$"); ax.set_ylabel("seed standard deviation of recovery fraction")
ax.set_title("Reproducibility of the spectral result\n%s" % SUB)
ax.set_xticks(WID_P); ax.legend(loc="best")
savefig(fig, "HA4_seed_spread")

# ---- printed summary -----------------------------------------------------------
print("\n=== recovery fraction at t=%.2f (mean +- sd over seeds) ===" % ANCHOR)
print("%4s %26s %10s %10s %6s" % ("n", "model", "mean", "sd", "runs"))
for n in WID_P:
    for m in ORDER:
        v = A[(A.n_feat == n) & (A.model == m)].phi_recovery
        if len(v): print("%4d %26s %10.4f %10.4f %6d" % (n, STYLE[m][1], v.mean(), v.std(), len(v)))
print("\n=== QAPINN vs c-PINN, head to head (budget-matched) ===")
for n in WID_P:
    q = A[(A.n_feat == n) & (A.model == "qapinn")].phi_model.mean()
    t_ = A[(A.n_feat == n) & (A.model == "twin")].phi_model.mean()
    if np.isfinite(q) and np.isfinite(t_) and t_ > 0:
        print("  n=%d  QAPINN %.4e   c-PINN %.4e   ratio %.3f" % (n, q, t_, q/t_))

## 4. Workstream B: what width costs

In [ ]:
# ---- HB1: gradient variance ----------------------------------------------------
fig, axs = plt.subplots(1, 2, figsize=(11.2, 4.3))
ax = axs[0]
for m in ORDER:
    s = GRAD[GRAD.model == m].sort_values("n_feat")
    if s.empty: continue
    col, lab, mk = STYLE[m]
    ax.semilogy(s.n_feat, s.variance, color=col, marker=mk, ms=6, lw=1.7, label=lab)
    x = s.n_feat.values.astype(float); y = s.variance.values
    ok = np.isfinite(y) & (y > 0)
    if ok.sum() >= 4:
        b, a = np.polyfit(x[ok], np.log(y[ok]), 1)
        ax.semilogy(x[ok], np.exp(a + b*x[ok]), color=col, ls="--", lw=1.0, alpha=0.6)
ax.set_xlabel("qubit count / feature width $n$")
ax.set_ylabel("variance of first-layer gradients")
ax.set_title("(a)  Gradient signal against width")
ax.set_xticks(WID_A); ax.legend(loc="best")

ax = axs[1]
for m in ORDER:
    s = TIME[TIME.model == m].sort_values("n_feat")
    if s.empty: continue
    col, lab, mk = STYLE[m]
    ax.semilogy(s.n_feat, s.sec_per_step, color=col, marker=mk, ms=6, lw=1.7, label=lab)
ax.set_xlabel("qubit count / feature width $n$")
ax.set_ylabel("seconds per optimisation step")
ax.set_title("(b)  Cost against width (400 collocation points)")
ax.set_xticks(WID_A); ax.legend(loc="best")
fig.suptitle("The two costs of width\n%s" % SUB, y=1.02)
plt.tight_layout(); savefig(fig, "HB1_gradient_and_cost")

print("\n=== fitted scaling ===")
for m in ORDER:
    s = GRAD[GRAD.model == m].sort_values("n_feat")
    t_ = TIME[TIME.model == m].sort_values("n_feat")
    for lab2, x, y in [("gradient", s.n_feat.values.astype(float), s.variance.values),
                       ("cost", t_.n_feat.values.astype(float), t_.sec_per_step.values)]:
        ok = np.isfinite(y) & (y > 0)
        if ok.sum() < 4: continue
        b, a = np.polyfit(x[ok], np.log(y[ok]), 1)
        # A perfectly flat series has zero variance in log(y), so R^2 is undefined
        # rather than poor. That is a real outcome here (classical cost does not change
        # with width), so it is labelled rather than printed as inf.
        den = np.sum((np.log(y[ok]) - np.log(y[ok]).mean())**2)
        if den <= 0:
            print("  %-24s %-9s slope %+.4f/qubit  factor %.2fx  R2 undefined (flat)  "
                  "total %.1fx" % (STYLE[m][1], lab2, b, np.exp(b), y[ok][-1]/y[ok][0]))
        else:
            r2 = 1 - np.sum((np.log(y[ok])-(a+b*x[ok]))**2)/den
            print("  %-24s %-9s slope %+.4f/qubit  factor %.2fx  R2 %.3f  total %.1fx"
                  % (STYLE[m][1], lab2, b, np.exp(b), r2, y[ok][-1]/y[ok][0]))

# ---- HB2: cost-benefit ----------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.4, 4.3))
gq = GRAD[GRAD.model == "qapinn"].sort_values("n_feat")
tq = TIME[TIME.model == "qapinn"].sort_values("n_feat")
ax.semilogy(gq.n_feat, gq.variance/gq.variance.iloc[0], color=C_Q, marker="o", ms=6, lw=1.8,
            label="gradient signal, relative to $n=3$")
ax.semilogy(tq.n_feat, tq.sec_per_step/tq.sec_per_step.iloc[0], color="#b8860b", marker="D",
            ms=6, lw=1.8, label="cost per step, relative to $n=3$")
ax.axhline(1.0, color="0.4", ls=":", lw=1.2)
ax.set_xlabel("qubit count $n$"); ax.set_ylabel("factor relative to $n=3$")
ax.set_title("Adding qubits: signal falls while cost rises\n%s" % SUB)
ax.set_xticks(WID_A); ax.legend(loc="best")
savefig(fig, "HB2_cost_benefit")

# ---- HB3: optimisation traces ---------------------------------------------------
ncol = 3; nrow = math.ceil(len(WID_A)/ncol)
fig, axs = plt.subplots(nrow, ncol, figsize=(4.1*ncol, 3.2*nrow),
                        sharex=True, sharey=True, squeeze=False)
for i, n in enumerate(WID_A):
    ax = axs[i//ncol][i % ncol]
    for m in ORDER:
        s = HIST[(HIST.n_feat == n) & (HIST.model == m)]
        if s.empty: continue
        col, lab, _ = STYLE[m]
        for _, g in s.groupby("tag"):
            g = g.sort_values("step")
            ax.semilogy(g.step, np.maximum(g.loss, 1e-12), color=col, alpha=0.6, lw=0.9)
        ax.plot([], [], color=col, label="%s (%d)" % (lab, s.tag.nunique()))
    ax.set_title("$n=%d$" % n)
axs[0][0].legend(loc="upper right", fontsize=7.2)
for j in range(ncol): axs[nrow-1][j].set_xlabel("recorded step")
for i2 in range(nrow): axs[i2][0].set_ylabel("training loss")
for i in range(len(WID_A), nrow*ncol): axs[i//ncol][i % ncol].axis("off")
fig.suptitle("Optimisation traces, one line per seed\n%s" % SUB, y=1.005)
plt.tight_layout(); savefig(fig, "HB3_loss_traces")

# ---- HB4: accuracy ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.2, 4.3))
acc = ALL.groupby(["model", "n_feat", "tag"]).l2_field.first().reset_index()
for m in ORDER:
    s = acc[acc.model == m]
    if s.empty: continue
    col, lab, mk = STYLE[m]
    g = s.groupby("n_feat").l2_field
    series_plot(ax, g.mean().index, g.mean().values, g.std().values, m, lab)
    ax.scatter(s.n_feat + dx(m), s.l2_field, color=col, s=13, alpha=0.30, linewidths=0)
ax.set_yscale("log"); ax.set_xlabel("width $n$")
ax.set_ylabel(r"relative $L^2$ error against the analytical solution")
ax.set_title("Accuracy against width (q8 has a shorter training budget)\n%s"
             % SUB)
ax.set_xticks(WID_A); ax.legend(loc="best")
savefig(fig, "HB4_accuracy")

print("\n=== accuracy, relative L2 (mean over seeds) ===")
print("%4s %26s %12s %12s" % ("n", "model", "mean", "sd"))
for n in WID_A:
    for m in ORDER:
        v = acc[(acc.n_feat == n) & (acc.model == m)].l2_field
        if len(v): print("%4d %26s %12.4e %12.4e" % (n, STYLE[m][1], v.mean(), v.std()))

## 5. Workstream C: information flow and neuron-level structure

In [ ]:
CK = CKA[CKA.in_primary == True].copy()
CK["first"] = first_col(CK)
CKALL = CKA.copy(); CKALL["first"] = first_col(CKALL)

# ---- HC1: CKA first layer to output, against time --------------------------------
ncol = 3; nrow = math.ceil(len(WID_P)/ncol)
fig, axs = plt.subplots(nrow, ncol, figsize=(4.1*ncol, 3.2*nrow),
                        sharex=True, sharey=True, squeeze=False)
for i, n in enumerate(WID_P):
    ax = axs[i//ncol][i % ncol]
    for m in ORDER:
        s = CK[(CK.n_feat == n) & (CK.model == m)]
        if s.empty: continue
        col, lab, _ = STYLE[m]
        for _, g in s.groupby("tag"):
            g = g.sort_values("t"); ax.plot(g.t, g["first"], color=col, alpha=0.6, lw=1.0)
        ax.plot([], [], color=col, label="%s (%d)" % (lab, s.tag.nunique()))
    ax.set_title("$n=%d$" % n); ax.set_ylim(-0.02, 1.02)
axs[0][0].legend(loc="lower left", fontsize=7.2)
for j in range(ncol): axs[nrow-1][j].set_xlabel("$t$")
for i2 in range(nrow): axs[i2][0].set_ylabel("CKA(first layer, output)")
for i in range(len(WID_P), nrow*ncol): axs[i//ncol][i % ncol].axis("off")
fig.suptitle("Information surviving from the first layer into the prediction\n%s" % SUB,
             y=1.005)
plt.tight_layout(); savefig(fig, "HC1_cka_vs_time")

# ---- HC2: CKA against width -------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.2, 4.3))
for m in ORDER:
    s = CK[CK.model == m]
    if s.empty: continue
    col, lab, mk = STYLE[m]
    g = s.groupby("n_feat")["first"]
    series_plot(ax, g.mean().index, g.mean().values, g.std().values, m, lab)
ax.set_ylim(-0.02, 1.02); ax.set_xlabel("width $n$")
ax.set_ylabel("CKA(first layer, output)")
ax.set_title("Does the first layer matter more as it widens?\n%s" % SUB)
ax.set_xticks(WID_P); ax.legend(loc="best")
savefig(fig, "HC2_cka_vs_width")

# ---- HC3: CKA by depth --------------------------------------------------------------
def depth_cols(df):
    cs = [c for c in df.columns if c.startswith("cka_") and c.endswith("_out")]
    f = [c for c in cs if "L0" in c]
    r = sorted([c for c in cs if "L0" not in c], key=lambda s: int(s.split("_")[1][1:]))
    return f + r
fig, ax = plt.subplots(figsize=(7.6, 4.3))
nmax = 0
for m in ORDER:
    s = CK[CK.model == m]
    if s.empty: continue
    col, lab, mk = STYLE[m]
    cs = [c for c in depth_cols(s) if s[c].notna().any()]
    nmax = max(nmax, len(cs))
    series_plot(ax, np.arange(len(cs)), [s[c].mean() for c in cs],
                [s[c].std() for c in cs], m, lab)
ax.set_xticks(range(nmax))
ax.set_xticklabels(["first layer\n(quantum / front)"] + ["L%d" % i for i in range(1, nmax)],
                   fontsize=8)
ax.set_ylabel("CKA(layer, output)"); ax.set_ylim(-0.02, 1.05)
ax.set_xlabel("layer (input side on the left)")
ax.set_title("Where the output representation is formed\n%s" % SUB)
ax.legend(loc="best"); savefig(fig, "HC3_cka_by_depth")

# ---- HC4: layer-to-layer CKA matrix, the item deferred at nu=0.05 -------------------
LL = [c for c in CKA.columns if c.startswith("ckaLL_")]
if LL:
    SEED0 = 1234 if 1234 in set(CKA.seed.unique()) else sorted(CKA.seed.unique())[0]
    nshow = 5 if 5 in WID_P else WID_P[len(WID_P)//2]
    present = [m for m in ORDER if not CKA[(CKA.model == m) & (CKA.n_feat == nshow)].empty]
    fig, axs = plt.subplots(1, len(present), figsize=(4.3*len(present), 4.0), squeeze=False)
    for k, m in enumerate(present):
        s = CKA[(CKA.model == m) & (CKA.n_feat == nshow) & (CKA.seed == SEED0)]
        if s.empty: axs[0][k].axis("off"); continue
        names = []
        for c in LL:
            a, b = c[6:].split("__")
            for nm in (a, b):
                if nm not in names: names.append(nm)
        names = ([x for x in names if x.startswith("L0")] +
                 sorted([x for x in names if not x.startswith("L0")],
                        key=lambda z: int(z[1:])))
        M = np.eye(len(names))
        for c in LL:
            a, b = c[6:].split("__")
            if a in names and b in names and s[c].notna().any():
                v = s[c].mean(); M[names.index(a), names.index(b)] = v
                M[names.index(b), names.index(a)] = v
        ax = axs[0][k]
        im = ax.imshow(M, cmap="viridis", vmin=0, vmax=1)
        lbl = ["first"] + names[1:]
        ax.set_xticks(range(len(names))); ax.set_xticklabels(lbl, rotation=90, fontsize=7)
        ax.set_yticks(range(len(names))); ax.set_yticklabels(lbl, fontsize=7)
        ax.set_title(STYLE[m][1], fontsize=9.5, color=STYLE[m][0]); ax.grid(False)
    fig.colorbar(im, ax=axs[0, :].tolist(), fraction=0.025, label="layer-to-layer CKA")
    fig.suptitle("Layer-to-layer similarity, $n=%d$, seed %d, mean over 26 slices\n%s"
                 % (nshow, SEED0, SUB), y=1.03)
    savefig(fig, "HC4_layer_to_layer_cka")
else:
    print("[skip] HC4: no ckaLL_ columns found")

## 6. Workstream C: per-neuron figures

In [ ]:
NP = NEU[NEU.in_primary == True] if "in_primary" in NEU.columns else NEU
SEED0 = 1234 if 1234 in set(NEU.seed.unique()) else sorted(NEU.seed.unique())[0]
T_REF = float(sorted(NEU.t.unique())[len(sorted(NEU.t.unique()))//2])
print("per-neuron figures: seed %d, reference slice t=%.2f" % (SEED0, T_REF))

# ---- HC5: per-neuron spread, layers 2 and 3, every width ---------------------------
for n in WID_A:
    s = NEU[(NEU.n_feat == n) & (NEU.seed == SEED0) & (np.isclose(NEU.t, T_REF))
            & (NEU.layer.isin(["L2", "L3"]))]
    if s.empty: continue
    kinds = [m for m in ORDER if m in set(s.model.unique())]
    fig, axs = plt.subplots(2, len(kinds), figsize=(4.2*len(kinds), 5.9),
                            squeeze=False, sharey="row")
    for c_, m in enumerate(kinds):
        col, lab, _ = STYLE[m]
        for r_, L in enumerate(["L2", "L3"]):
            ax = axs[r_][c_]
            g = s[(s.model == m) & (s.layer == L)].sort_values("neuron")
            ax.bar(g.neuron, g["std"], color=col, alpha=0.88)
            ax.set_title("%s, layer %s" % (lab, L), fontsize=9.5)
            ax.set_xlabel("neuron index")
            if c_ == 0: ax.set_ylabel("activation standard deviation")
    fig.suptitle("Per-neuron activation spread, $n=%d$, seed %d, $t=%.2f$\n%s"
                 % (n, SEED0, T_REF, SUB), y=1.01)
    plt.tight_layout(); savefig(fig, "HC5_neuron_spread_n%d" % n)

# ---- HC6: neuron against time, all 26 slices ---------------------------------------
for n in WID_A:
    base = NEU[(NEU.n_feat == n) & (NEU.seed == SEED0)]
    if base.empty: continue
    kinds = [m for m in ORDER if m in set(base.model.unique())]
    layers = ["FIRST", "L2", "L3"]
    fig, axs = plt.subplots(len(layers), len(kinds),
                            figsize=(4.3*len(kinds), 3.0*len(layers)), squeeze=False)
    _v = base[base.layer.isin(["L2", "L3"])]["std"].quantile(0.99)
    vmax = float(_v) if np.isfinite(_v) else 1.0
    for c_, m in enumerate(kinds):
        for r_, L in enumerate(layers):
            ax = axs[r_][c_]
            if L == "FIRST":
                g = base[(base.model == m) & (base.layer.str.startswith("L0"))]
                lname = g.layer.iloc[0] if not g.empty else "L0"
                _vv = g["std"].quantile(0.99) if not g.empty else np.nan
                vm = float(_vv) if np.isfinite(_vv) else vmax
            else:
                g = base[(base.model == m) & (base.layer == L)]; lname, vm = L, vmax
            if g.empty or not np.isfinite(vm) or vm <= 0: ax.axis("off"); continue
            M = g.pivot_table(index="neuron", columns="t", values="std")
            im = ax.imshow(M.values, aspect="auto", origin="lower", cmap="magma",
                           vmin=0, vmax=vm,
                           extent=[M.columns.min(), M.columns.max(), -0.5, M.index.max()+0.5])
            ax.set_title("%s, %s" % (STYLE[m][1], lname), fontsize=9.2)
            if r_ == len(layers)-1: ax.set_xlabel("$t$")
            if c_ == 0: ax.set_ylabel("neuron index")
            ax.grid(False)
            if L == "FIRST": fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    fig.colorbar(im, ax=axs[1:, :].ravel().tolist(), fraction=0.02,
                 label="activation standard deviation (rows 2-3)")
    fig.suptitle("Every neuron across all %d slices, $n=%d$, seed %d\n%s"
                 % (NEU.t.nunique(), n, SEED0, SUB), y=1.01)
    savefig(fig, "HC6_neuron_time_n%d" % n)

# ---- HC7: saturation by layer --------------------------------------------------------
def depth_layers(ls):
    f = [L for L in ls if L.startswith("L0")]
    r = sorted([L for L in ls if not L.startswith("L0")], key=lambda z: int(z[1:]))
    return f + r
fig, ax = plt.subplots(figsize=(7.6, 4.3))
nmax = 0
for m in ORDER:
    s = NEU[NEU.model == m]
    if s.empty: continue
    col, lab, mk = STYLE[m]
    ls = depth_layers(list(s.layer.unique())); nmax = max(nmax, len(ls))
    ax.plot(range(len(ls)), [s[s.layer == L].sat.mean() for L in ls],
            color=col, marker=mk, ms=6, lw=1.7, label=lab)
ax.set_xticks(range(nmax))
ax.set_xticklabels(["first layer\n(quantum / front)"] + ["L%d" % i for i in range(1, nmax)],
                   fontsize=8)
ax.set_ylabel(r"fraction of inputs with $|a|>0.95$"); ax.set_xlabel("layer")
ax.set_title("Neuron saturation by layer\n%s" % SUB)
ax.legend(loc="best"); savefig(fig, "HC7_saturation")

# ---- HC8: per-neuron contribution ------------------------------------------------------
fig, axs = plt.subplots(1, 2, figsize=(11.2, 4.3))
ax = axs[0]; w = 0.26
ls_all = depth_layers(list(NEU.layer.unique()))
gen = [L for L in ls_all if not L.startswith("L0")]
for k, m in enumerate(ORDER):
    s = NEU[NEU.model == m]
    if s.empty: continue
    col, lab, _ = STYLE[m]
    xs = np.arange(len(gen)) + (k-1)*w
    ax.bar(xs, [s[s.layer == L].cka_out.mean() for L in gen], w,
           yerr=[s[s.layer == L].cka_out.std() for L in gen],
           color=col, alpha=0.9, capsize=2, label=lab)
ax.set_xticks(range(len(gen))); ax.set_xticklabels(gen, fontsize=8)
ax.set_ylabel("mean per-neuron CKA with the output")
ax.set_title("(a)  Individual neuron contribution, classical layers")
ax.legend(loc="best")
ax = axs[1]
for m in ORDER:
    s = NEU[(NEU.model == m) & (NEU.layer.str.startswith("L0"))]
    if s.empty: continue
    col, lab, _ = STYLE[m]
    v = s.cka_out.dropna()
    if len(v) < 5: continue
    ax.hist(v, bins=40, histtype="step", lw=1.8, color=col, density=True,
            label="%s (%d units)" % (lab, len(v)))
ax.set_xlabel("per-neuron CKA with the output, first layer"); ax.set_ylabel("density")
ax.set_title("(b)  Distribution, first layer"); ax.legend(loc="best")
fig.suptitle("Which units drive the prediction\n%s" % SUB, y=1.02)
plt.tight_layout(); savefig(fig, "HC8_neuron_contribution")

## 7. Summary tables and manifest

In [ ]:
rows = []
A = WSA[(WSA.in_primary == True) & (np.isclose(WSA.t, ANCHOR))]
for m in ORDER:
    for n in sorted(A[A.model == m].n_feat.unique()):
        s = A[(A.model == m) & (A.n_feat == n)]
        g = GRAD[(GRAD.model == m) & (GRAD.n_feat == n)]
        t_ = TIME[(TIME.model == m) & (TIME.n_feat == n)]
        c = CKA[(CKA.model == m) & (CKA.n_feat == n)]
        c_first = first_col(c)
        rows.append(dict(model=m, n_feat=int(n), runs=int(s.tag.nunique()),
                         phi_recovery_mean=float(s.phi_recovery.mean()),
                         phi_recovery_sd=float(s.phi_recovery.std()),
                         phi_model_mean=float(s.phi_model.mean()),
                         l2_mean=float(s.groupby("tag").l2_field.first().mean()),
                         grad_var=float(g.variance.iloc[0]) if len(g) else np.nan,
                         sec_per_step=float(t_.sec_per_step.iloc[0]) if len(t_) else np.nan,
                         cka_first_mean=float(c_first.mean()),
                         sat_first=float(NEU[(NEU.model == m) & (NEU.n_feat == n)
                                             & (NEU.layer.str.startswith("L0"))].sat.mean())))
S = pd.DataFrame(rows)
S.to_csv(os.path.join(OUT, "phase3_hard_summary.csv"), index=False)
print(S.to_string(index=False))
print("\n[saved] phase3_hard_summary.csv")

figs = sorted(os.listdir(FIG))
json.dump(dict(note="phase3-hard-figures (Track A, Djabon)",
               nu="0.01/pi", anchor_slice=ANCHOR, analytical_phi_max=PHI_MAX,
               n_figures=len(figs), figures=figs,
               primary_widths=[int(x) for x in WID_P],
               all_widths=[int(x) for x in WID_A],
               caveats=[
                 "Every value is read from the builder's CSVs; nothing is recomputed and no "
                 "synthetic data is used.",
                 "q8 is excluded from WS-A and WS-C primary panels: its recorded budget is 703 "
                 "steps against 2016 at every other width. It is shown in WS-B panels, where "
                 "gradient variance and cost are measured at initialisation and are unaffected "
                 "by training length, and in HB3/HB4 where it is labelled.",
                 "At q3-q7 all three architectures share the same 2016-step budget, so the "
                 "head-to-head comparison at those widths is budget-matched.",
                 "Timing was measured on one machine at 400 collocation points and is "
                 "internally comparable only.",
               ]),
          open(os.path.join(OUT, "phase3_hard_figures_manifest.json"), "w"), indent=2)
print("\n%d figures in %s" % (len(figs), FIG))
for f in figs: print("   ", f)